In [4]:
import os
import pandas as pd
import pickle
from string import punctuation

from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import wordnet, stopwords
from nltk.classify import accuracy, NaiveBayesClassifier
from nltk.probability import FreqDist
from nltk.stem import WordNetLemmatizer

In [5]:
eng_stopwords = stopwords.words('english')
lemmatizer = WordNetLemmatizer()
MODEL_PATH = './model.pickle'
DATA_PATH = './financial_dataset.csv'

In [7]:
def getTag(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('R'):
        return 'r'
    elif tag.startswith('V'):
        return 'v'
    else:
        return 'n'
    
def lemmatizing(tokens):
    lemmatized = []
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        result = lemmatizer.lemmatize(word, getTag(tag))
        lemmatized.append(result)
    return lemmatized

def preprocessing(sentence):
    sentence = sentence.lower()
    tokens = word_tokenize(sentence)
    tokens = [token for token in tokens if token not in eng_stopwords]
    tokens = [token for token in tokens if token not in punctuation]
    tokens = [token for token in tokens if token.isalpha()]
    tokens = lemmatizing(tokens)
    return tokens

df = pd.read_csv(DATA_PATH)
x = df['Statement']
y = df['Sentiment']
allSentence = ' '.join(x)
allToken = preprocessing(allSentence)
freqDist = FreqDist(allToken)

def extract_feature(sentence):
    featured = {}
    for word in freqDist.keys():
        featured[word] = (word in sentence)
    return featured

feature_set = [(extract_feature(preprocessing(sentence)), sentiment)
               for sentence, sentiment in zip(x, y)
]

split = int(len(feature_set) * 0.8)
train_data = feature_set[:split]
test_data = feature_set[split:]

def trainModel():
    model = NaiveBayesClassifier.train(train_data)
    model.show_most_informative_features(7)
    acc = accuracy(model, test_data)
    print(f"accuracy: {acc}")

    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(model, f)
    return model

In [13]:
def show_pos_tag(sentence):
    tokens = preprocessing(sentence)
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        print(f"{word}: {tag}")
    return tokens

def show_ant_syn(tokens):
    for token in tokens:
        synonyms = []
        antonyms = []
        for syn in wordnet.synsets(token):
            for lemma in syn.lemmas():
                synonyms.append(lemma.name())
                for ant in lemma.antonyms():
                    antonyms.append(ant.name())

        if synonyms:
            print(f"Synonym: {synonyms}")
        else:
            print("No synonym")
        if antonyms:
            print(f"Antonym: {antonyms}")
        else:
            print("No antonym")
    
def predictStatement(model, tokens):
    feature = extract_feature(tokens)
    prediction = model.classify(feature)
    print(f"Prediction: {prediction}")

def analyzeStatement(model, sentence):
    tokens = show_pos_tag(sentence)
    show_ant_syn(tokens)
    predictStatement(model, tokens)

In [15]:
sentence = ''
while True:
    choice = input(">> ")
    if choice == '1':
        sentence = input("Input statement: ")
        if len(sentence.split()) < 2:
            break
        else:
            if os.path.exists(MODEL_PATH):
                with open(MODEL_PATH, 'rb') as f:
                    model = pickle.load(f)
            else:
                model = trainModel()

    elif choice == '2':
        analyzeStatement(model, sentence)
    elif choice == '3':
        break
    else:
        print("Invalid input")

budiono: NN
siregar: NN
No synonym
No antonym
No synonym
No antonym
Prediction: positive
